# YOLOv11m Segmentation 추가학습 파이프라인

## 사전 준비 (Google Drive에 업로드 해놓기)
경로: `내 드라이브/CCATFARM_data/`
- `kkat_dataset/` 폴더 (gofile에서 다운)
- `ccat/` 폴더 (CVAT export - images/ + annotations.xml)
- `yolo11m_seg_aug_20260707_155305_best.pt` (모델 파일)

## 런타임 설정
런타임 → 런타임 유형 변경 → **GPU (T4)** 선택

## 셀 순서대로 실행하면 됩니다

## 1. GPU 확인 + 패키지 설치

In [ ]:
!nvidia-smi
!pip install ultralytics albumentations -q

## 2. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. GitHub 클론 (스크립트 가져오기)

In [ ]:
!git clone https://github.com/Samingyeong/CCATFARM.git /content/CCATFARM
!ls /content/CCATFARM/Model-jetson/

## 4. 데이터 합치기 (01_merge_dataset.py)
- ccat (CVAT XML) → YOLO seg 포맷 변환
- kkat_dataset + 변환된 ccat → dataset_all/original에 합침

In [ ]:
!python /content/CCATFARM/Model-jetson/01_merge_dataset.py \
  --ccat /content/drive/MyDrive/CCATFARM_data/ccat \
  --kkat /content/drive/MyDrive/CCATFARM_data/kkat_dataset \
  --output /content/dataset_all

## 5. 1차 추가학습 (03_train_step1.py)
- 155305 모델 + ccat 24장(원본)으로 fine-tuning
- → 1차 모델 생성

In [ ]:
!python /content/CCATFARM/Model-jetson/03_train_step1.py \
  --model /content/drive/MyDrive/CCATFARM_data/yolo11m_seg_aug_20260707_155305_best.pt \
  --dataset /content/dataset_all \
  --output /content/runs

## 6. 데이터 증강 (02_augment_dataset.py)
- 원본 전체 5배 증강 (Albumentations)
- train/val 분할 + data.yaml 생성

In [ ]:
!python /content/CCATFARM/Model-jetson/02_augment_dataset.py \
  --dataset /content/dataset_all

## 7. 2차 추가학습 (03_train_step2.py)
- 1차 모델 + 전체 데이터(원본+증강)로 학습
- → 최종 모델 생성

In [ ]:
!python /content/CCATFARM/Model-jetson/03_train_step2.py \
  --model /content/runs/step1_ccat_finetune/weights/best.pt \
  --dataset /content/dataset_all \
  --output /content/runs

## 8. 테스트 (04_test.py)

In [ ]:
!python /content/CCATFARM/Model-jetson/04_test.py \
  --model /content/runs/step2_full_finetune/weights/best.pt \
  --dataset /content/dataset_all \
  --output /content/runs

## 9. 결과 시각화

In [ ]:
from IPython.display import Image, display
import glob, os

# 학습 곡선
results_img = '/content/runs/step2_full_finetune/results.png'
if os.path.exists(results_img):
    print('=== 학습 곡선 ===')
    display(Image(results_img, width=800))

# 예측 결과
pred_images = sorted(glob.glob('/content/runs/test_results/*.jpg'))[:6]
for img_path in pred_images:
    print(os.path.basename(img_path))
    display(Image(img_path, width=600))

## 10. 최종 모델 Drive에 저장

In [ ]:
import shutil

# best.pt 저장
src = '/content/runs/step2_full_finetune/weights/best.pt'
dst = '/content/drive/MyDrive/CCATFARM_data/finetune_final_best.pt'
shutil.copy2(src, dst)
print(f'최종 모델 저장 완료: {dst}')

# 1차 모델도 저장
src1 = '/content/runs/step1_ccat_finetune/weights/best.pt'
dst1 = '/content/drive/MyDrive/CCATFARM_data/finetune_step1_best.pt'
shutil.copy2(src1, dst1)
print(f'1차 모델 저장 완료: {dst1}')

## 완료!
- `finetune_final_best.pt` → 최종 모델 (Drive에 저장됨)
- `finetune_step1_best.pt` → 1차 모델 (Drive에 저장됨)
- 이 파일을 Jetson에 배포하면 됨
- ONNX/TensorRT 변환은 별도 진행